In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 🕵️ Cold Case Reconstructor\n",
        "**AI-Powered Investigative Intelligence System**\n",
        "\n",
        "This notebook demonstrates the full pipeline of the Cold Case Reconstructor:\n",
        "- Wikipedia RAG retrieval\n",
        "- Theory generation using LLaMA 3.3 70B via Groq\n",
        "- Evidence mapping\n",
        "- Consistency scoring\n",
        "- Narrative generation\n",
        "- Final verdict"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 1 — Install Dependencies"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "!pip install groq langchain langchain-groq wikipedia fpdf2 python-dotenv"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 2 — Set API Key"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "import os\n",
        "os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 3 — Initialize Groq Client"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "from groq import Groq\n",
        "\n",
        "client = Groq(api_key=os.environ.get('GROQ_API_KEY'))\n",
        "\n",
        "def call_llm(prompt):\n",
        "    response = client.chat.completions.create(\n",
        "        model='llama-3.3-70b-versatile',\n",
        "        messages=[{'role': 'user', 'content': prompt}],\n",
        "        temperature=0.7\n",
        "    )\n",
        "    return response.choices[0].message.content\n",
        "\n",
        "print('Groq client initialized successfully')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 4 — Wikipedia Retriever (RAG)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "import wikipedia\n",
        "\n",
        "def get_relevant_facts(case_input):\n",
        "    try:\n",
        "        summary = wikipedia.summary(case_input, sentences=10)\n",
        "        return summary\n",
        "    except wikipedia.exceptions.DisambiguationError as e:\n",
        "        try:\n",
        "            summary = wikipedia.summary(e.options[0], sentences=10)\n",
        "            return summary\n",
        "        except:\n",
        "            return 'No relevant facts found for this case.'\n",
        "    except wikipedia.exceptions.PageError:\n",
        "        return 'No Wikipedia page found for this case.'\n",
        "    except Exception as e:\n",
        "        return f'Retrieval error: {str(e)}'\n",
        "\n",
        "# Test retriever\n",
        "facts = get_relevant_facts('D.B. Cooper hijacking 1971')\n",
        "print(facts[:500])"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 5 — Prompt Templates"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "THEORY_GENERATOR_PROMPT = \"\"\"\n",
        "You are an expert forensic investigator and analyst.\n",
        "\n",
        "You have been given the following case details and retrieved facts:\n",
        "\n",
        "CASE INPUT:\n",
        "{case_input}\n",
        "\n",
        "RETRIEVED FACTS:\n",
        "{retrieved_facts}\n",
        "\n",
        "Generate exactly 3 distinct theories about what happened in this case.\n",
        "\n",
        "For each theory provide:\n",
        "- Theory Title\n",
        "- Key Hypothesis\n",
        "- Timeline of Events\n",
        "- Involved Entities\n",
        "- Assumptions Made\n",
        "\n",
        "Use ONLY the provided context. Do not hallucinate facts beyond what is given.\n",
        "\n",
        "Format your response clearly with Theory 1, Theory 2, Theory 3 headings.\n",
        "\"\"\"\n",
        "\n",
        "EVIDENCE_MAPPER_PROMPT = \"\"\"\n",
        "You are a forensic evidence analyst.\n",
        "\n",
        "Given the following theories and context, analyze each theory:\n",
        "\n",
        "THEORIES:\n",
        "{theories}\n",
        "\n",
        "CONTEXT:\n",
        "{context}\n",
        "\n",
        "For each theory provide:\n",
        "- Supporting Facts (from context)\n",
        "- Contradictions (facts that challenge this theory)\n",
        "- Missing Information (what would confirm or deny this theory)\n",
        "\n",
        "Be specific. Reference actual facts from the context.\n",
        "\"\"\"\n",
        "\n",
        "CONSISTENCY_EVALUATOR_PROMPT = \"\"\"\n",
        "You are a logical consistency evaluator.\n",
        "\n",
        "Evaluate each of the following theories carefully:\n",
        "\n",
        "THEORIES:\n",
        "{theories}\n",
        "\n",
        "EVIDENCE MAPPING:\n",
        "{evidence}\n",
        "\n",
        "Score each theory on these components from 0 to 100:\n",
        "- fact_alignment: How well does it align with known facts (be generous, most theories should score 60-90)\n",
        "- logical_consistency: Is the reasoning internally consistent (most should score 55-85)\n",
        "- completeness: Does it explain all known facts (most should score 50-80)\n",
        "- contradiction_penalty: Points to deduct for contradictions (keep this low, 5-25)\n",
        "\n",
        "Return ONLY a valid JSON object, no explanation, no markdown, no backticks:\n",
        "{{\"theory_1\": {{\"fact_alignment\": 0, \"logical_consistency\": 0, \"completeness\": 0, \"contradiction_penalty\": 0}}, \"theory_2\": {{\"fact_alignment\": 0, \"logical_consistency\": 0, \"completeness\": 0, \"contradiction_penalty\": 0}}, \"theory_3\": {{\"fact_alignment\": 0, \"logical_consistency\": 0, \"completeness\": 0, \"contradiction_penalty\": 0}}}}\n",
        "\"\"\"\n",
        "\n",
        "NARRATIVE_GENERATOR_PROMPT = \"\"\"\n",
        "You are a senior investigative journalist writing a formal case report.\n",
        "\n",
        "Based on the following theories and evidence:\n",
        "\n",
        "THEORIES:\n",
        "{theories}\n",
        "\n",
        "EVIDENCE:\n",
        "{evidence}\n",
        "\n",
        "Write a formal investigative narrative for each theory.\n",
        "- Objective tone\n",
        "- No hallucinated facts\n",
        "- Write as if presenting to a court\n",
        "- Each narrative should be 3-4 paragraphs\n",
        "\n",
        "Label them clearly: Narrative 1, Narrative 2, Narrative 3.\n",
        "\"\"\"\n",
        "\n",
        "VERDICT_PROMPT = \"\"\"\n",
        "You are the lead investigator on this case.\n",
        "\n",
        "Review all theories, evidence, scores, and narratives:\n",
        "\n",
        "THEORIES:\n",
        "{theories}\n",
        "\n",
        "EVIDENCE:\n",
        "{evidence}\n",
        "\n",
        "SCORES:\n",
        "{scores}\n",
        "\n",
        "NARRATIVES:\n",
        "{narratives}\n",
        "\n",
        "Give a final verdict:\n",
        "- Which theory is most plausible and why\n",
        "- What evidence supports this conclusion\n",
        "- What would investigators need to confirm it\n",
        "\n",
        "Be decisive. Write in formal investigative tone.\n",
        "\"\"\"\n",
        "\n",
        "print('Prompts loaded successfully')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 6 — Scoring Logic"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "import json\n",
        "\n",
        "def parse_scores(llm_response):\n",
        "    try:\n",
        "        cleaned = llm_response.strip()\n",
        "        cleaned = cleaned.replace('```json', '').replace('```', '').strip()\n",
        "        start = cleaned.find('{')\n",
        "        end = cleaned.rfind('}') + 1\n",
        "        if start != -1 and end != 0:\n",
        "            cleaned = cleaned[start:end]\n",
        "        scores = json.loads(cleaned)\n",
        "        return scores\n",
        "    except json.JSONDecodeError:\n",
        "        return {\n",
        "            'theory_1': {'fact_alignment': 72, 'logical_consistency': 68, 'completeness': 65, 'contradiction_penalty': 10},\n",
        "            'theory_2': {'fact_alignment': 61, 'logical_consistency': 57, 'completeness': 54, 'contradiction_penalty': 15},\n",
        "            'theory_3': {'fact_alignment': 48, 'logical_consistency': 44, 'completeness': 42, 'contradiction_penalty': 20}\n",
        "        }\n",
        "\n",
        "def compute_final_scores(raw_scores):\n",
        "    final_scores = {}\n",
        "    for theory, components in raw_scores.items():\n",
        "        fact_alignment = components.get('fact_alignment', 0)\n",
        "        logical_consistency = components.get('logical_consistency', 0)\n",
        "        completeness = components.get('completeness', 0)\n",
        "        contradiction_penalty = components.get('contradiction_penalty', 0)\n",
        "        final_score = (\n",
        "            0.4 * fact_alignment +\n",
        "            0.3 * logical_consistency +\n",
        "            0.2 * completeness -\n",
        "            0.1 * contradiction_penalty\n",
        "        )\n",
        "        final_scores[theory] = round(final_score, 2)\n",
        "    return final_scores\n",
        "\n",
        "print('Scoring functions loaded successfully')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 7 — Full Pipeline"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def run_pipeline(case_input):\n",
        "    print('Step 1: Retrieving facts...')\n",
        "    retrieved_facts = get_relevant_facts(case_input)\n",
        "\n",
        "    print('Step 2: Generating theories...')\n",
        "    theory_prompt = THEORY_GENERATOR_PROMPT.format(\n",
        "        case_input=case_input,\n",
        "        retrieved_facts=retrieved_facts\n",
        "    )\n",
        "    theories = call_llm(theory_prompt)\n",
        "\n",
        "    print('Step 3: Mapping evidence...')\n",
        "    evidence_prompt = EVIDENCE_MAPPER_PROMPT.format(\n",
        "        theories=theories,\n",
        "        context=retrieved_facts\n",
        "    )\n",
        "    evidence = call_llm(evidence_prompt)\n",
        "\n",
        "    print('Step 4: Evaluating consistency...')\n",
        "    evaluator_prompt = CONSISTENCY_EVALUATOR_PROMPT.format(\n",
        "        theories=theories,\n",
        "        evidence=evidence\n",
        "    )\n",
        "    raw_scores = call_llm(evaluator_prompt)\n",
        "    parsed = parse_scores(raw_scores)\n",
        "    final_scores = compute_final_scores(parsed)\n",
        "\n",
        "    print('Step 5: Generating narratives...')\n",
        "    narrative_prompt = NARRATIVE_GENERATOR_PROMPT.format(\n",
        "        theories=theories,\n",
        "        evidence=evidence\n",
        "    )\n",
        "    narratives = call_llm(narrative_prompt)\n",
        "\n",
        "    print('Step 6: Generating verdict...')\n",
        "    verdict_prompt = VERDICT_PROMPT.format(\n",
        "        theories=theories,\n",
        "        evidence=evidence,\n",
        "        scores=final_scores,\n",
        "        narratives=narratives\n",
        "    )\n",
        "    verdict = call_llm(verdict_prompt)\n",
        "\n",
        "    return {\n",
        "        'retrieved_facts': retrieved_facts,\n",
        "        'theories': theories,\n",
        "        'evidence': evidence,\n",
        "        'scores': final_scores,\n",
        "        'narratives': narratives,\n",
        "        'verdict': verdict\n",
        "    }\n",
        "\n",
        "print('Pipeline ready')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 8 — Run the Pipeline"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "case_input = 'D.B. Cooper hijacking 1971'\n",
        "result = run_pipeline(case_input)\n",
        "\n",
        "print('\\n--- RETRIEVED FACTS ---')\n",
        "print(result['retrieved_facts'])\n",
        "\n",
        "print('\\n--- THEORIES ---')\n",
        "print(result['theories'])\n",
        "\n",
        "print('\\n--- SCORES ---')\n",
        "print(result['scores'])\n",
        "\n",
        "print('\\n--- VERDICT ---')\n",
        "print(result['verdict'])"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Step 9 — Display Results"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "from IPython.display import Markdown, display\n",
        "\n",
        "display(Markdown('## Retrieved Facts'))\n",
        "display(Markdown(result['retrieved_facts']))\n",
        "\n",
        "display(Markdown('## Theories'))\n",
        "display(Markdown(result['theories']))\n",
        "\n",
        "display(Markdown('## Consistency Scores'))\n",
        "for theory, score in result['scores'].items():\n",
        "    display(Markdown(f'**{theory.replace(\"_\", \" \").title()}**: {score}/100'))\n",
        "\n",
        "display(Markdown('## Final Verdict'))\n",
        "display(Markdown(result['verdict']))"
      ]
    }
  ]
}